In [30]:
#import function used to split the dataset for train and test
from sklearn.model_selection import train_test_split as tts

#import linear regression 
from sklearn.linear_model import LinearRegression 

#import evaluation metrics so we can judge the model afterwards
from sklearn.metrics import mean_squared_error as ms_error, r2_score

#import numpy to calc sqrt mse
import numpy as np
import pandas as pd

In [31]:
dataset = pd.read_csv("data/Filtered_Experimental_Records.csv")





def extract_site_composition(ions, coefficients, group):

    # Start by giving every element at this site a value of 0
    # If an element is not present in a particular perovskite,
    # its value will stay as 0
    composition = {
        element: 0.0 for element in group
    }

    # If either the ion column or coefficient column is missing,
    # return the default values of 0
    if pd.isna(ions) or pd.isna(coefficients):
        return composition

    # Split the ion string at each semicolon
    # Example:
    # "Cs; FA; MA" becomes ["Cs", "FA", "MA"]
    ion_list = [
        ion.strip()
        for ion in str(ions).split(";")
    ]

    # Split the coefficient string at each semicolon
    # and convert each value into a number
    # Example:
    # "0.05; 0.85; 0.15" becomes [0.05, 0.85, 0.15]
    coefficient_list = [
        float(value.strip())
        for value in str(coefficients).split(";")
    ]

    # Match each ion with the coefficient in the same position
    # Example:
    # Cs -> 0.05
    # FA -> 0.85
    # MA -> 0.15
    for ion, coefficient in zip(ion_list, coefficient_list):

        # Only save species that are expected at this site
        if ion in composition:
            composition[ion] = coefficient

    # Return the finished numerical composition for this site
    return composition





# --------------------------------------------------
# 2. DEFINE THE POSSIBLE SPECIES AT EACH SITE
# --------------------------------------------------

# A-site species
# MA and FA are molecular ions, but they are treated as
# individual composition features in the ML model
a_site_species = [
    "Cs",
    "FA",
    "MA"
]

# B-site species
b_site_species = [
    "Pb",
    "Sn"
]

# X-site species
# Your dataset calls this the "c_ions" column
x_site_species = [
    "Cl",
    "Br",
    "I"
]



# --------------------------------------------------
# 3. EXTRACT THE A-SITE COMPOSITION
# --------------------------------------------------

# Apply the function to every row of the dataset
a_composition = dataset.apply(
    lambda row: extract_site_composition(
        row["Perovskite_composition_a_ions"],
        row["Perovskite_composition_a_ions_coefficients"],
        a_site_species
    ),
    axis=1
)

# Convert the resulting dictionaries into normal DataFrame columns
a_composition = pd.DataFrame(
    a_composition.tolist(),
    index=dataset.index
)



# --------------------------------------------------
# 4. EXTRACT THE B-SITE COMPOSITION
# --------------------------------------------------

b_composition = dataset.apply(
    lambda row: extract_site_composition(
        row["Perovskite_composition_b_ions"],
        row["Perovskite_composition_b_ions_coefficients"],
        b_site_species
    ),
    axis=1
)

# Convert the results into DataFrame columns
b_composition = pd.DataFrame(
    b_composition.tolist(),
    index=dataset.index
)






# --------------------------------------------------
# 5. EXTRACT THE X-SITE COMPOSITION
# --------------------------------------------------

x_composition = dataset.apply(
    lambda row: extract_site_composition(
        row["Perovskite_composition_c_ions"],
        row["Perovskite_composition_c_ions_coefficients"],
        x_site_species
    ),
    axis=1
)

# Convert the results into DataFrame columns
x_composition = pd.DataFrame(
    x_composition.tolist(),
    index=dataset.index
)





# --------------------------------------------------
# 6. COMBINE ALL COMPOSITION FEATURES
# --------------------------------------------------

# Join the A-, B-, and X-site numerical features together
composition_features = pd.concat(
    [
        a_composition,
        b_composition,
        x_composition
    ],
    axis=1
)

# Display the first few rows
display(composition_features.head())



,Cs,FA,MA,Pb,Sn,Cl,Br,I
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
1,1.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0
3,1.0,0.0,0.0,0.0,1.0,0.0,1.0,2.0
4,1.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0


In [32]:
# Add the extracted numerical composition features
# onto the original dataset
dataset = pd.concat(
    [
        dataset,
        composition_features
    ],
    axis=1
)

# Check that the new columns are now present
print(dataset.columns.tolist())

['Ref_ID', 'Ref_DOI_number', 'Perovskite_composition_a_ions', 'Perovskite_composition_a_ions_coefficients', 'Perovskite_composition_b_ions', 'Perovskite_composition_b_ions_coefficients', 'Perovskite_composition_c_ions', 'Perovskite_composition_c_ions_coefficients', 'Perovskite_composition_short_form', 'Perovskite_band_gap', 'Cell_architecture', 'Cell_area_measured', 'Substrate_stack_sequence', 'ETL_stack_sequence', 'HTL_stack_sequence', 'Backcontact_stack_sequence', 'JV_default_PCE', 'JV_default_Jsc', 'JV_default_Voc', 'JV_default_FF', 'Cs', 'FA', 'MA', 'Pb', 'Sn', 'Cl', 'Br', 'I']


In [33]:
#dataset = pd.read_csv("data/Filtered_Experimental_Records.csv")
x = dataset [["Cs",
        "MA",
        "FA",
        "Pb",
        "Sn",
        "Br",
        "I",
        "Cl"]]
y = dataset["Perovskite_band_gap"]



In [34]:
#8:2 split for training:test

x_train, x_test, y_train, y_test = tts(
    x,y,test_size = 0.2, random_state =67)

In [35]:
model = LinearRegression()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)


In [36]:
#compare with test

mse = ms_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MSE:", mse, "RMSE", rmse, "R^2", r2)



MSE: 0.0025151994074118023 RMSE 0.050151763751754556 R^2 0.7757770984990997
